## **ETL del dataset de modelado**

### **Objetivo**

En esta etapa se construye el dataset analítico que será utilizado luego para el EDA y el modelado.

La unidad de análisis sugerida en la exploración previa es el siniestro vial, por lo tanto el dataset final deberá tener una fila por `id_siniestro`.


### **Carga de datos**

Se cargan nuevamente los datasets originales para comenzar el proceso de transformación y construcción del dataset analítico.

In [1]:
import pandas as pd
import numpy as np


# Carga de datasets

hechos = pd.read_excel(
    "../data_raw/siniestros_viales_hechos.xlsx",
    sheet_name="HECHOS"
)

victimas = pd.read_excel(
    "../data_raw/siniestros_viales_victimas.xlsx",
    sheet_name="VICTIMAS"
)

### **Validación de consistencia entre Hechos y Víctimas**

Antes de construir la variable objetivo, se verificará si la cantidad de víctimas fatales registrada en la tabla Hechos  coincide con la información disponible en la tabla Víctimas.

Esta validación permitirá confirmar la consistencia entre ambas fuentes y definir cuál será utilizada para la construcción de la variable objetivo.

In [2]:
# Normalizar categorías de gravedad

victimas["gravedad_victima_normalizada"] = (
    victimas["GRAVEdad_victima"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Cantidad de víctimas fatales por siniestro según VICTIMAS

fatalidades_victimas = (
    victimas
    .assign(
        victima_fatal=lambda df: (
            df["gravedad_victima_normalizada"] == "MORTAL"
        ).astype(int)
    )
    .groupby("id_siniestro", as_index=False)
    .agg(
        victimas_fatales_victimas=("victima_fatal", "sum")
    )
)

# Comparar contra HECHOS

comparacion = hechos[
    ["id_siniestro", "numero_victimas_mortal_siniestro"]
].merge(
    fatalidades_victimas,
    on="id_siniestro",
    how="left"
)

comparacion["victimas_fatales_victimas"] = (
    comparacion["victimas_fatales_victimas"]
    .fillna(0)
    .astype(int)
)

comparacion["coincide"] = (
    comparacion["numero_victimas_mortal_siniestro"]
    == comparacion["victimas_fatales_victimas"]
)

comparacion["coincide"].value_counts()

coincide
True    65818
Name: count, dtype: int64

**Observación**

La cantidad de víctimas mortales registrada en la tabla Hechos coincide exactamente con la reconstruida a partir de la tabla Víctimas para los 65.818 siniestros analizados.

Dado que ambas fuentes presentan consistencia total, se utilizará la variable `numero_victimas_mortal_siniestro` de la tabla Hechos para construir la variable objetivo, evitando transformaciones adicionales innecesarias.

### **Construcción de la variable objetivo**

De acuerdo con el objetivo del proyecto, se construye una variable binaria que indique si un siniestro tuvo al menos una víctima fatal.

Se utilizará la variable `numero_victimas_mortal_siniestro` de la tabla Hechos, cuya consistencia fue previamente validada contra la información disponible en la tabla Víctimas.

In [3]:
# Variable objetivo

hechos["siniestro_fatal"] = (
    hechos["numero_victimas_mortal_siniestro"] > 0
).astype(int)

hechos["siniestro_fatal"].value_counts()


siniestro_fatal
0    65129
1      689
Name: count, dtype: int64

In [4]:
hechos["siniestro_fatal"].value_counts(normalize=True) * 100

siniestro_fatal
0    98.953174
1     1.046826
Name: proportion, dtype: float64

**Observación**

La variable objetivo quedó construida correctamente.

Se identificaron 689 siniestros con al menos una víctima fatal sobre un total de 65.818 siniestros registrados.

Esto representa aproximadamente el 1,05% de los casos, mientras que el 98,95% corresponde a siniestros sin víctimas fatales.

Por lo tanto, el problema presenta un marcado desbalance de clases, aspecto que deberá considerarse durante la etapa de modelado y evaluación de los algoritmos.

### **Identificación de variables candidatas para el modelado**

Se revisan las variables disponibles en las tablas Hechos y Víctimas para identificar cuáles podrían formar parte del dataset analítico final.

El objetivo es detectar variables equivalentes entre ambas tablas, así como información disponible únicamente a nivel de víctima que eventualmente podría incorporarse al nivel siniestro.

In [5]:
print("Columnas de Hechos:")
for col in hechos.columns:
    print("-", col)

print("\n" + "="*60 + "\n")

print("Columnas de Víctimas:")
for col in victimas.columns:
    print("-", col)

Columnas de Hechos:
- id_siniestro
- numero_total_de_victimas
- numero_victimas_leve_siniestro
- numero_victimas_grave_siniestro
- numero_victimas_mortal_siniestro
- fecha_siniestro
- anio_siniestro
- mes_siniestro
- dia_siniestro
- hora_siniestro
- rango_horario
- direccion_normalizada_siniestro
- comuna_siniestro
- tipo_de_via_siniestro
- geocodificacion_plana
- longitud_siniestro
- latitud_siniestro
- participantes_siniestro
- modo_desplazamiento_victima
- contraparte_siniestro
- gravedad_siniestro
- siniestro_fatal


Columnas de Víctimas:
- id_siniestro
- fecha_siniestro
- anio_siniestro
- modo_desplazamiento_victima
- sexo_victima
- edad_victima
- GRAVEdad_victima
- rol_victima
- fecha_fallecimiento_victima
- gravedad_victima_normalizada


**Observación**

El listado de columnas muestra que varias variables de Víctimas ya tienen una variable equivalente en Hechos, como `id_siniestro`, `fecha_siniestro`, `anio_siniestro` y `modo_desplazamiento_victima`.

Dado que la unidad de análisis del proyecto es el siniestro, se priorizará el uso de las variables ya disponibles en Hechos cuando representen información equivalente a nivel agregado.

Las variables de Víctimas que podrían aportar información adicional son principalmente `sexo_victima`, `edad_victima` y `rol_victima`, aunque requieren una evaluación previa de calidad y una posible agregación a nivel siniestro.

### Calidad de las variables exclusivas de Víctimas

Las variables `sexo_victima`, `edad_victima` y `rol_victima` sólo se encuentran disponibles en la tabla Víctimas.

Antes de evaluar su incorporación al dataset analítico, se analizará la proporción de registros con valor `SD` (Sin Datos).

In [6]:
variables_victimas = [
    "sexo_victima",
    "edad_victima",
    "rol_victima"
]

for columna in variables_victimas:

    cantidad_sd = (
        victimas[columna]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("SD")
        .sum()
    )

    porcentaje_sd = (
        cantidad_sd / len(victimas) * 100
    )

    print(f"{columna}")
    print(f"SD: {cantidad_sd}")
    print(f"% SD: {porcentaje_sd:.2f}%")
    print("-" * 40)

sexo_victima
SD: 17094
% SD: 22.73%
----------------------------------------
edad_victima
SD: 23372
% SD: 31.08%
----------------------------------------
rol_victima
SD: 60136
% SD: 79.98%
----------------------------------------


**Observación**

Las variables exclusivas de la tabla Víctimas presentan distintos niveles de completitud.

- `sexo_victima` registra un 22,73% de valores SD.
- `edad_victima` registra un 31,08% de valores SD.
- `rol_victima` registra un 79,98% de valores SD.

La variable `rol_victima` presenta un nivel de faltantes muy elevado, por lo que inicialmente se considera una candidata a descarte para el dataset de modelado.

Por el contrario, `sexo_victima` y `edad_victima` conservan una proporción importante de información válida y serán analizadas con mayor detalle antes de decidir su incorporación.

### **Exploración de sexo_victima y edad_victima**

Se analizan las categorías presentes en las variables `sexo_victima` y `edad_victima` para evaluar su calidad y posible incorporación al dataset de modelado.

In [7]:
(
    victimas["sexo_victima"]
    .astype(str)
    .str.strip()
    .value_counts(dropna=False)
)

sexo_victima
M     38221
F     19878
SD    17094
Name: count, dtype: int64

In [8]:
victimas["edad_victima"].describe()

count     75193
unique      106
top          SD
freq      23296
Name: edad_victima, dtype: object

**Observaciones**

La variable `sexo_victima` presenta tres categorías:

- M: 38.221 registros.
- F: 19.878 registros.
- SD: 17.094 registros.

Los valores SD representan aproximadamente el 22,73% de los registros, mientras que el 77,27% restante cuenta con información válida.

A pesar de la presencia de valores faltantes, la variable conserva una proporción significativa de datos útiles, por lo que inicialmente se considera candidata para futuras etapas de análisis.

La variable `edad_victima` se encuentra almacenada como tipo `object` y combina valores numéricos con el valor textual `SD`.

Por este motivo, no puede ordenarse directamente como variable numérica. Será necesario crear una versión transformada de la edad, convirtiendo los valores no numéricos a nulos para poder analizarla correctamente.

In [9]:
# Conversión de edad_victima a formato numérico

victimas["edad_victima_num"] = pd.to_numeric(
    victimas["edad_victima"],
    errors="coerce"
)

victimas["edad_victima_num"].describe()

count    51821.000000
mean        37.939966
std         15.815169
min          0.000000
25%         26.000000
50%         35.000000
75%         47.000000
max        116.000000
Name: edad_victima_num, dtype: float64

**Observaciones**

La variable `edad_victima` fue transformada a formato numérico, convirtiendo los valores `SD` a nulos.

Luego de la transformación se observó que:

- Se dispone de información válida para 51.821 víctimas.
- La edad promedio es de aproximadamente 38 años.
- La mediana es de 35 años.
- La edad mínima registrada es 0 años.
- La edad máxima registrada es 116 años.

No se detectaron valores evidentemente imposibles, por lo que la variable se considera apta para futuras etapas de análisis y modelado.

### **Revisión de tipos de datos del dataset base Hechos**

La tabla Hechos será utilizada como base para la construcción del dataset analítico.

Antes de comenzar las transformaciones, se revisan los tipos de datos actuales para identificar variables que requieran tratamiento durante el proceso ETL.

In [10]:
hechos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65818 entries, 0 to 65817
Data columns (total 22 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   id_siniestro                      65818 non-null  object
 1   numero_total_de_victimas          65818 non-null  object
 2   numero_victimas_leve_siniestro    65818 non-null  int64 
 3   numero_victimas_grave_siniestro   65818 non-null  int64 
 4   numero_victimas_mortal_siniestro  65818 non-null  int64 
 5   fecha_siniestro                   65818 non-null  object
 6   anio_siniestro                    65818 non-null  int64 
 7   mes_siniestro                     65818 non-null  int64 
 8   dia_siniestro                     65818 non-null  int64 
 9   hora_siniestro                    65741 non-null  object
 10  rango_horario                     65741 non-null  object
 11  direccion_normalizada_siniestro   52918 non-null  object
 12  comuna_siniestro  

**Observaciones**

La revisión de tipos de datos muestra que varias variables ya se encuentran correctamente almacenadas como numéricas, especialmente las relacionadas con cantidades de víctimas y componentes temporales (`anio_siniestro`, `mes_siniestro`, `dia_siniestro`).

Sin embargo, se identifican algunas variables que requerirán transformación durante el proceso ETL:

- `fecha_siniestro` se encuentra almacenada como texto.
- `hora_siniestro` se encuentra almacenada como texto y presenta valores faltantes.
- `longitud_siniestro` y `latitud_siniestro` se encuentran almacenadas como texto.
- `numero_total_de_victimas` se encuentra almacenada como texto a pesar de representar una cantidad.

Asimismo, se observan valores faltantes en variables geográficas y descriptivas como `comuna_siniestro`, `tipo_de_via_siniestro`, `direccion_normalizada_siniestro` y las coordenadas geográficas.

### Transformación de fecha_siniestro

In [11]:
# Convertimos la fecha al tipo datetime para facilitar futuros análisis temporales

hechos["fecha_siniestro"] = pd.to_datetime(
    hechos["fecha_siniestro"],
    errors="coerce"
)

# Verificamos el tipo de dato resultante

print("Tipo de dato:")
print(hechos["fecha_siniestro"].dtype)

# Comprobamos si aparecieron fechas inválidas

print("\nValores nulos:")
print(hechos["fecha_siniestro"].isna().sum())

Tipo de dato:
datetime64[ns]

Valores nulos:
0


A partir de la fecha del siniestro se genera una variable que identifica el día de la semana en que ocurrió el hecho.

Esta información podría aportar contexto temporal adicional al modelo.

In [12]:
# Generamos el día de la semana a partir de la fecha

hechos["dia_semana"] = hechos["fecha_siniestro"].dt.day_name()

# Revisamos la distribución obtenida

hechos["dia_semana"].value_counts()

dia_semana
Friday       11120
Thursday     10678
Wednesday    10391
Tuesday      10319
Monday        9670
Saturday      7791
Sunday        5849
Name: count, dtype: int64

**Observaciones**

Se generó correctamente la variable `dia_semana` a partir de `fecha_siniestro`.

La distribución de registros muestra presencia de siniestros en todos los días de la semana, con una mayor frecuencia entre lunes y viernes y una menor cantidad durante los fines de semana.

La variable se considera candidata para futuras etapas de análisis y modelado.

### **Revisión y transformación de hora_siniestro y rango_horario**

La variable `hora_siniestro` contiene la hora en la que ocurrió el hecho y podría aportar información relevante para el análisis y el modelado.

Se revisará su formato actual y la presencia de valores faltantes antes de realizar transformaciones adicionales.

In [13]:
# Revisamos algunos valores de la variable hora

print("Primeros valores:")
print(hechos["hora_siniestro"].dropna().head(10))

print("\nCantidad de nulos:")
print(hechos["hora_siniestro"].isna().sum())

Primeros valores:
0    01:55:00
1    18:50:00
2    13:40:00
3    07:15:00
4    12:24:00
5    04:30:00
6    09:00:00
7    07:30:00
8    08:05:00
9    09:36:00
Name: hora_siniestro, dtype: object

Cantidad de nulos:
77


In [14]:
# Revisamos las categorías presentes en rango_horario

hechos["rango_horario"].value_counts(dropna=False)


rango_horario
17     4748
16     4610
14     4458
18     4421
15     4361
13     4181
12     4116
19     3718
11     3384
20     3231
9      3141
10     3135
8      2960
21     2708
22     2224
7      2150
0      1901
23     1617
6      1317
1       947
5       754
2       693
3       508
4       454
NaN      77
SD        4
Name: count, dtype: int64

Se verifica si la variable `rango_horario` representa efectivamente la hora del día agrupada en intervalos horarios equivalente a la variable `hora_siniestro`

In [15]:
# Revisamos algunos registros para comparar ambas variables

hechos[
    ["hora_siniestro", "rango_horario"]
].head(20)

,hora_siniestro,rango_horario
0,01:55:00,1
1,18:50:00,18
2,13:40:00,13
3,07:15:00,7
4,12:24:00,12
5,04:30:00,4
6,09:00:00,9
7,07:30:00,7
8,08:05:00,8
9,09:36:00,9


**Observaciones**

Se verificó que la variable `rango_horario` representa la hora del siniestro expresada como un valor entero entre 0 y 23.

Por lo tanto, ambas variables contienen esencialmente la misma información, aunque con distinto nivel de detalle.

Dado que `rango_horario` ya se encuentra discretizada y resulta más simple de utilizar en etapas posteriores, se considera candidata a reemplazar a `hora_siniestro` para fines analíticos y de modelado.

### **Revisión de coordenadas geográficas**

Las variables `latitud_siniestro` y `longitud_siniestro` contienen la ubicación geográfica del hecho.

Se revisará su formato actual para determinar si requieren transformación a tipo numérico.

In [16]:
# Revisamos algunos valores de las coordenadas

print("Longitud:")
print(hechos["longitud_siniestro"].dropna().head(10))

print("\nLatitud:")
print(hechos["latitud_siniestro"].dropna().head(10))

Longitud:
0     -58.44351
1    -58.490436
2    -58.524662
3    -58.528413
4    -58.404748
5    -58.398225
6    -58.408911
7    -58.519116
8    -58.439392
9    -58.371488
Name: longitud_siniestro, dtype: object

Latitud:
0    -34.669125
1    -34.583403
2    -34.603255
3    -34.650156
4    -34.648387
5    -34.604579
6    -34.559658
7    -34.589282
8    -34.588108
9    -34.598416
Name: latitud_siniestro, dtype: object


**Observaciones**

Las variables `longitud_siniestro` y `latitud_siniestro` contienen coordenadas geográficas válidas expresadas con punto decimal.

Aunque actualmente se encuentran almacenadas como tipo `object`, los valores observados presentan un formato numérico consistente, por lo que podrán transformarse a tipo numérico sin necesidad de tareas de limpieza previas.

In [17]:
# Convertimos las coordenadas a formato numérico

hechos["longitud_siniestro"] = pd.to_numeric(
    hechos["longitud_siniestro"],
    errors="coerce"
)

hechos["latitud_siniestro"] = pd.to_numeric(
    hechos["latitud_siniestro"],
    errors="coerce"
)

# Verificamos el resultado de la conversión

print("Tipo longitud:", hechos["longitud_siniestro"].dtype)
print("Nulos longitud:", hechos["longitud_siniestro"].isna().sum())

print("\nTipo latitud:", hechos["latitud_siniestro"].dtype)
print("Nulos latitud:", hechos["latitud_siniestro"].isna().sum())

Tipo longitud: float64
Nulos longitud: 3031

Tipo latitud: float64
Nulos latitud: 3031


**Observaciones**

Las variables `longitud_siniestro` y `latitud_siniestro` fueron convertidas correctamente a formato numérico.

Luego de la transformación se identificaron 3.031 valores nulos en ambas variables, cifra superior a la observada inicialmente durante la fase de comprensión de los datos.

Esto sugiere la existencia de registros sin coordenadas válidas o con formatos que no pudieron convertirse a valores numéricos. Y esos valores eran SD (652 valores en dataset original)

### **Revisión de variables categóricas principales**

In [18]:
# Variables categóricas relevantes para el análisis

variables_categoricas = [
    "comuna_siniestro",
    "tipo_de_via_siniestro",
    "participantes_siniestro",
    "modo_desplazamiento_victima",
    "contraparte_siniestro"
]

for columna in variables_categoricas:

    print(f"\n{'='*60}")
    print(columna)
    print(f"{'='*60}")

    print(
        hechos[columna]
        .value_counts(dropna=False)
        .head(15)
    )


comuna_siniestro
comuna_siniestro
Comuna 1     5790
Comuna 15    4361
Comuna 3     3994
Comuna 4     3720
Comuna 9     3653
Comuna 14    3648
Comuna 12    3623
Comuna 7     3537
Comuna 13    3249
Comuna 10    3133
NaN          3017
Comuna 11    2997
Comuna 5     2619
Comuna 8     2480
Comuna 6     2255
Name: count, dtype: int64

tipo_de_via_siniestro
tipo_de_via_siniestro
AVENIDA          30525
CALLE            18794
NaN              12227
AV. GRAL. PAZ     2916
AUTOPISTA         1041
SD                 315
Name: count, dtype: int64

participantes_siniestro
participantes_siniestro
MOTO-AUTO                                15178
SD-SD                                    10348
PEATON-AUTO                               4491
BICICLETA-AUTO                            4057
AUTO-AUTO                                 3877
TRANSPORTE PUBLICO-TRANSPORTE PUBLICO     2539
MOTO-UTILITARIO                           1750
MOTO-SD                                   1669
PEATON-MOTO                        

**Observaciones**

Las variables categóricas presentan distintos niveles de calidad y completitud.

- `comuna_siniestro` contiene 3.017 valores faltantes.
- `tipo_de_via_siniestro` contiene 12.227 valores nulos y 315 registros con valor `SD`.
- `participantes_siniestro` presenta una elevada frecuencia de la categoría `SD-SD`.
- `modo_desplazamiento_victima` contiene 10.928 registros con valor `SD`.
- `contraparte_siniestro` contiene 14.066 registros con valor `SD`.

Además, se identificó una inconsistencia de formato en `contraparte_siniestro`, donde coexisten las categorías `AUTO` y `auto`, lo que requerirá normalización.

In [19]:
# Unificamos el formato de las categorías

hechos["contraparte_siniestro"] = (
    hechos["contraparte_siniestro"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Revisamos nuevamente las categorías más frecuentes

hechos["contraparte_siniestro"].value_counts(dropna=False).head(15)

contraparte_siniestro
AUTO                  29996
SD                    14067
TRANSPORTE PUBLICO     6035
UTILITARIO             3575
MOTO                   3496
TAXI                   2624
CAMION                 2579
OBJETO FIJO            1521
MOVIL                   804
BICICLETA               390
MULTIPLE                347
OTRO                    204
CAMIONETA                74
MONOPATIN                57
PEATON                   36
Name: count, dtype: int64

In [20]:
# Revisamos algunos ejemplos de las variables geográficas complementarias

print("direccion_normalizada_siniestro")
print(
    hechos["direccion_normalizada_siniestro"]
    .dropna()
    .head(10)
)

print("\n" + "="*60 + "\n")

print("geocodificacion_plana")
print(
    hechos["geocodificacion_plana"]
    .dropna()
    .head(10)
)

direccion_normalizada_siniestro
1                     DE LOS CONSTITUYENTES AV. y HABANA
18                               COLONIA AV. y USPALLATA
112                 IRIGOYEN, BERNARDO DE y SAN JUAN AV.
124                   OBLIGADO RAFAEL, AV.COSTANERA 4600
152    CASTILLO, RAMON S., PRES. AV. y CALLE 12 (NO O...
181                 CASTAÃ‘ARES AV. y MORENO, PERITO AV.
182                      ESTADO DE ISRAEL AV. y PRINGLES
205                        SANTA FE AV. y RODRIGUEZ PE?A
219                PAZ, GRAL. AV. y ACHA, MARIANO, GRAL.
220                 RAWSON DE DELLEPIANE, ELVIRA AV. 150
Name: direccion_normalizada_siniestro, dtype: object


geocodificacion_plana
0    POINT (101813.84712503915943671 95578.55507230...
1    POINT (97510.27137648081406951 105087.83369881...
2    POINT (94371.44237885093025398 102884.19365105...
3    POINT (94030.76669932194636203 97681.071761248...
4    POINT (105367.86665769023238681 97877.75085908...
5    POINT (105968.98286849579017144 102737.17

### **Evaluación de completitud de variables candidatas**

Se analiza el nivel de completitud de las principales variables candidatas para el modelado, considerando tanto valores nulos como categorías utilizadas para representar ausencia de información (SD).

Esto permitirá identificar variables que requieran tratamiento especial y evaluar su viabilidad para formar parte del dataset analítico final.

In [21]:
# Evaluamos la completitud de las principales variables candidatas

variables = [
    "comuna_siniestro",
    "tipo_de_via_siniestro",
    "participantes_siniestro",
    "modo_desplazamiento_victima",
    "contraparte_siniestro",
    "latitud_siniestro",
    "longitud_siniestro"
]

for columna in variables:

    nulos = hechos[columna].isna().sum()

    sd = (
        hechos[columna]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("SD")
        .sum()
    )

    porcentaje = ((nulos + sd) / len(hechos)) * 100

    print(f"\n{'='*50}")
    print(columna)
    print(f"{'='*50}")
    print(f"Nulos: {nulos}")
    print(f"SD: {sd}")
    print(f"Sin información: {porcentaje:.2f}%")


comuna_siniestro
Nulos: 3017
SD: 535
Sin información: 5.40%

tipo_de_via_siniestro
Nulos: 12227
SD: 315
Sin información: 19.06%

participantes_siniestro
Nulos: 0
SD: 0
Sin información: 0.00%

modo_desplazamiento_victima
Nulos: 0
SD: 10928
Sin información: 16.60%

contraparte_siniestro
Nulos: 0
SD: 14067
Sin información: 21.37%

latitud_siniestro
Nulos: 3031
SD: 0
Sin información: 4.61%

longitud_siniestro
Nulos: 3031
SD: 0
Sin información: 4.61%


**Observaciones**

Las variables candidatas presentan distintos niveles de completitud.

- `participantes_siniestro` presenta una completitud prácticamente total.
- `comuna_siniestro`, `latitud_siniestro` y `longitud_siniestro` muestran menos del 6% de registros sin información.
- `modo_desplazamiento_victima` presenta un 16,60% de registros sin información.
- `tipo_de_via_siniestro` presenta un 19,06% de registros sin información.
- `contraparte_siniestro` presenta el mayor nivel de ausencia de información entre las variables analizadas (21,37%).

A pesar de estas diferencias, ninguna de las variables evaluadas presenta niveles de ausencia que justifiquen su descarte inmediato, por lo que continuarán siendo consideradas para la construcción del dataset analítico.

### **Evaluación del tratamiento de valores SD**

Varias variables categóricas utilizan el valor SD (Sin Datos) para representar ausencia de información.

Antes de continuar con el proceso ETL, se analizará si estos valores deben conservarse como una categoría propia o transformarse a valores nulos.

In [22]:
# Revisamos la presencia de valores SD en las principales variables categóricas

variables_sd = [
    "tipo_de_via_siniestro",
    "participantes_siniestro",
    "modo_desplazamiento_victima",
    "contraparte_siniestro"
]

for columna in variables_sd:

    sd = (
        hechos[columna]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("SD")
        .sum()
    )

    print(f"{columna}: {sd}")

tipo_de_via_siniestro: 315
participantes_siniestro: 0
modo_desplazamiento_victima: 10928
contraparte_siniestro: 14067


**Decisión metodológica**

Los valores `SD` (Sin Datos) se conservarán como una categoría propia en las variables categóricas.

Esta decisión se fundamenta en que la ausencia de información puede constituir una característica relevante del registro y su eliminación podría provocar pérdida de información o reducir innecesariamente la cantidad de observaciones disponibles para el análisis.

Los valores nulos reales (`NaN`) continuarán tratándose de manera independiente.

### **Evaluación de incorporación de variables provenientes de Víctimas**

Las variables edad_victima_num y sexo_victima contienen información potencialmente relevante, pero se encuentran registradas a nivel víctima.

Se analizará la cantidad de víctimas por siniestro para determinar la viabilidad de incorporar estas variables al dataset analítico cuya unidad de análisis es el siniestro.

In [23]:
# Analizamos la cantidad de víctimas asociadas a cada siniestro

victimas_por_siniestro = (
    victimas
    .groupby("id_siniestro")
    .size()
)

# Resumen estadístico

print("Resumen estadístico")
print(victimas_por_siniestro.describe())

print("\nDistribución de víctimas por siniestro")
print(victimas_por_siniestro.value_counts().sort_index())

Resumen estadístico
count    65818.000000
mean         1.142438
std          0.537284
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         26.000000
dtype: float64

Distribución de víctimas por siniestro
1     59053
2      5294
3       911
4       303
5       135
6        61
7        23
8        19
9         3
10        5
13        4
14        1
15        1
16        2
18        1
19        1
26        1
Name: count, dtype: int64


**Observaciones**

La unidad de análisis del proyecto es el siniestro, mientras que la tabla Víctimas registra información a nivel individual.

El análisis realizado muestra que el 89,72% de los siniestros (59.053 de 65.818) presenta una única víctima asociada.

Sin embargo, existe un conjunto menor de siniestros con múltiples víctimas, alcanzando un máximo de 26 víctimas registradas para un mismo hecho.

Esto implica que cualquier variable proveniente de la tabla Víctimas requeriría definir criterios de agregación para poder incorporarse al dataset analítico basado en siniestros.

In [24]:
# Evaluamos cuántos siniestros tienen información utilizable de edad

edad_por_siniestro = (
    victimas
    .groupby("id_siniestro")["edad_victima_num"]
    .apply(lambda x: x.notna().any())
)

print("Edad informada")
print(edad_por_siniestro.value_counts())

print("\n" + "="*50 + "\n")

# Evaluamos cuántos siniestros tienen información utilizable de sexo

sexo_por_siniestro = (
    victimas
    .groupby("id_siniestro")["sexo_victima"]
    .apply(
        lambda x: (
            x.astype(str)
             .str.upper()
             .isin(["M", "F"])
             .any()
        )
    )
)

print("Sexo informado")
print(sexo_por_siniestro.value_counts())

Edad informada
edad_victima_num
True     45259
False    20559
Name: count, dtype: int64


Sexo informado
sexo_victima
True     50830
False    14988
Name: count, dtype: int64


**Observaciones**

Se evaluó la disponibilidad de información proveniente de la tabla Víctimas a nivel siniestro.

Se observó que:

- 45.259 siniestros (68,76%) poseen al menos una edad informada.
- 50.830 siniestros (77,23%) poseen al menos un sexo informado.

Si bien la cobertura de estas variables resulta significativa, su incorporación al dataset analítico requeriría definir criterios de agregación debido a la existencia de siniestros con múltiples víctimas.

Por este motivo, se pospone la decisión de incorporar estas variables hasta la etapa de selección final de variables.

### **Evaluación de variables relacionadas con cantidad de víctimas**

Las variables relacionadas con cantidades de víctimas podrían aportar información relevante para el modelado, pero también podrían estar describiendo consecuencias del siniestro muy cercanas a la variable objetivo.

Se analizará su distribución para determinar su posible incorporación o exclusión.


In [25]:
# Revisamos la distribución de las variables relacionadas con víctimas

variables_victimas = [
    "numero_total_de_victimas",
    "numero_victimas_leve_siniestro",
    "numero_victimas_grave_siniestro",
    "numero_victimas_mortal_siniestro"
]

for columna in variables_victimas:

    print(f"\n{'='*60}")
    print(columna)
    print(f"{'='*60}")

    print(
        hechos[columna]
        .value_counts(dropna=False)
        .head(15)
    )


numero_total_de_victimas
numero_total_de_victimas
1     55776
2      5295
SD     3276
3       911
4       304
5       134
6        61
7        23
8        19
10        5
13        4
9         3
16        2
18        1
14        1
Name: count, dtype: int64

numero_victimas_leve_siniestro
numero_victimas_leve_siniestro
1     56142
2      4971
0      3322
3       852
4       287
5       130
6        57
7        23
8        16
13        5
9         4
10        3
16        3
26        1
15        1
Name: count, dtype: int64

numero_victimas_grave_siniestro
numero_victimas_grave_siniestro
0     62812
1      2871
2       111
3        18
4         3
5         1
8         1
10        1
Name: count, dtype: int64

numero_victimas_mortal_siniestro
numero_victimas_mortal_siniestro
0    65129
1      675
2       14
Name: count, dtype: int64


Son variables que describen un hecho o situación que se produce como consecuencia del accidente y no como causa, así que se excluíran como variables funcionales

### **Construcción del dataset base para modelado**

A partir de las decisiones tomadas durante el proceso ETL, se construye un dataset base que contiene únicamente las variables seleccionadas para las etapas posteriores de análisis y modelado.

Las variables descartadas permanecerán disponibles en los datasets originales para análisis complementarios, pero no formarán parte del dataset principal de trabajo.

In [26]:

# Construimos el dataset base para las siguientes etapas del proyecto

df = hechos[
    [
        "siniestro_fatal",
        "anio_siniestro",
        "mes_siniestro",
        "dia_siniestro",
        "dia_semana",
        "rango_horario",
        "comuna_siniestro",
        "tipo_de_via_siniestro",
        "participantes_siniestro",
        "modo_desplazamiento_victima",
        "contraparte_siniestro",
        "latitud_siniestro",
        "longitud_siniestro"
    ]
].copy()

# Verificamos dimensiones y estructura del dataset resultante

print("Shape:")
print(df.shape)

print("\nColumnas:")
print(df.columns.tolist())

Shape:
(65818, 13)

Columnas:
['siniestro_fatal', 'anio_siniestro', 'mes_siniestro', 'dia_siniestro', 'dia_semana', 'rango_horario', 'comuna_siniestro', 'tipo_de_via_siniestro', 'participantes_siniestro', 'modo_desplazamiento_victima', 'contraparte_siniestro', 'latitud_siniestro', 'longitud_siniestro']


**Observaciones**

Las variables relacionadas con cantidades de víctimas describen consecuencias del siniestro ya ocurrido.

En particular, `numero_victimas_mortal_siniestro` fue utilizada para construir la variable objetivo `siniestro_fatal`, por lo que no debe utilizarse como variable predictora.

También se observa que `numero_total_de_victimas` contiene valores `SD`, mientras que las variables parciales de víctimas leves, graves y mortales presentan valores numéricos. Por este motivo, y dado que estas variables están muy vinculadas al resultado del hecho, se decide no incluirlas como variables predictoras del dataset principal.

Estas variables podrán utilizarse más adelante para análisis descriptivo, pero no formarán parte del conjunto de variables candidatas para modelado.

### **Matriz preliminar de decisiones ETL**

A partir de las revisiones realizadas, se consolidan las decisiones preliminares sobre las variables disponibles.

Esta matriz permite dejar documentado qué variables se conservarán en el dataset principal, cuáles quedan en evaluación y cuáles no serán utilizadas como predictoras.


| Variable | Decisión preliminar | Motivo |
|---|---|---|
| id_siniestro | Conservar como soporte | Permite trazabilidad de los registros |
| siniestro_fatal | Conservar | Variable objetivo |
| fecha_siniestro | Conservar como soporte | Fecha transformada; útil para análisis temporal |
| anio_siniestro | Conservar | Variable temporal |
| mes_siniestro | Conservar | Variable temporal |
| dia_siniestro | Conservar | Variable temporal |
| dia_semana | Conservar | Variable derivada de `fecha_siniestro` |
| rango_horario | Conservar | Representa la hora del siniestro |
| comuna_siniestro | Conservar | Variable geográfica con buena completitud |
| tipo_de_via_siniestro | Conservar | Variable descriptiva del entorno vial |
| participantes_siniestro | Conservar | Variable relevante sobre participantes del hecho |
| modo_desplazamiento_victima | Conservar | Variable relevante sobre la víctima |
| contraparte_siniestro | Conservar | Variable relevante sobre contraparte; fue normalizada |
| latitud_siniestro | Conservar | Variable geográfica transformada a numérica |
| longitud_siniestro | Conservar | Variable geográfica transformada a numérica |
| edad_victima_num | En evaluación | Requiere agregación desde Víctimas a nivel siniestro |
| sexo_victima | En evaluación | Requiere agregación desde Víctimas a nivel siniestro |
| hora_siniestro | No usar como predictor | Redundante con `rango_horario` |
| direccion_normalizada_siniestro | No usar como predictor | Alta cardinalidad y problemas de codificación |
| geocodificacion_plana | No usar como predictor | Redundante con latitud/longitud |
| numero_total_de_victimas | No usar como predictor | Describe consecuencia del siniestro; contiene `SD` |
| numero_victimas_leve_siniestro | No usar como predictor | Describe consecuencia del siniestro |
| numero_victimas_grave_siniestro | No usar como predictor | Describe consecuencia del siniestro |
| numero_victimas_mortal_siniestro | No usar como predictor | Base de la variable objetivo |
| gravedad_siniestro | No usar como predictor | Muy cercana a la variable objetivo |
| GRAVEdad_victima | No usar como predictor | Variable de Víctimas directamente asociada a gravedad |
| rol_victima | No usar como predictor | Alto porcentaje de valores `SD` |
| fecha_fallecimiento_victima | No usar como predictor | Potencial fuga de información |
| gravedad_victima_normalizada | No exportar | Variable auxiliar temporal |

### **Evaluación de agregación de edad y sexo a nivel siniestro**

Las variables `edad_victima_num` y `sexo_victima` pertenecen a la tabla Víctimas, cuya unidad de análisis es la víctima individual.

Como el dataset principal del proyecto tendrá una fila por siniestro, se evalúa si estas variables pueden agregarse razonablemente al nivel de `id_siniestro`.

In [27]:
# Analizamos la cantidad de víctimas asociadas a cada siniestro
# para evaluar la complejidad de agregar variables de Víctimas

victimas_por_siniestro = (
    victimas
    .groupby("id_siniestro")
    .size()
)

print("Resumen estadístico:")
print(victimas_por_siniestro.describe())

print("\nDistribución de víctimas por siniestro:")
print(victimas_por_siniestro.value_counts().sort_index())

Resumen estadístico:
count    65818.000000
mean         1.142438
std          0.537284
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         26.000000
dtype: float64

Distribución de víctimas por siniestro:
1     59053
2      5294
3       911
4       303
5       135
6        61
7        23
8        19
9         3
10        5
13        4
14        1
15        1
16        2
18        1
19        1
26        1
Name: count, dtype: int64


**Observaciones**

El 89,72% de los siniestros tiene una única víctima asociada (59.053 de 65.818 registros).

Sin embargo, existen siniestros con múltiples víctimas, alcanzando un máximo de 26 víctimas en un mismo hecho.

Esto implica que las variables provenientes de Víctimas requieren un criterio de agregación para poder incorporarse al dataset principal.

In [28]:
# Evaluamos cuántos siniestros tienen al menos una edad válida

edad_por_siniestro = (
    victimas
    .groupby("id_siniestro")["edad_victima_num"]
    .apply(lambda x: x.notna().any())
)

print("Edad informada:")
print(edad_por_siniestro.value_counts())

print("\n" + "="*50 + "\n")

# Evaluamos cuántos siniestros tienen al menos un sexo informado

sexo_por_siniestro = (
    victimas
    .groupby("id_siniestro")["sexo_victima"]
    .apply(
        lambda x: (
            x.astype(str)
             .str.upper()
             .isin(["M", "F"])
             .any()
        )
    )
)

print("Sexo informado:")
print(sexo_por_siniestro.value_counts())

Edad informada:
edad_victima_num
True     45259
False    20559
Name: count, dtype: int64


Sexo informado:
sexo_victima
True     50830
False    14988
Name: count, dtype: int64


**Observaciones**

Se observa que 45.259 siniestros tienen al menos una edad válida y 50.830 siniestros tienen al menos un sexo informado.

Esto muestra que ambas variables tienen una cobertura relevante a nivel siniestro. Por lo tanto, no se descartan en esta etapa y se evaluará una agregación simple para incorporarlas al dataset principal.

In [29]:
# Analizamos posibles agregaciones de edad a nivel siniestro

edad_agregada = (
    victimas
    .groupby("id_siniestro")["edad_victima_num"]
    .agg(
        cantidad_edades="count",
        edad_promedio="mean",
        edad_minima="min",
        edad_maxima="max"
    )
)

edad_agregada.head(20)

,cantidad_edades,edad_promedio,edad_minima,edad_maxima
id_siniestro,,,,
HC-2019-0014291,1,25.000000,25.0,25.0
HC-2019-0016448,1,32.000000,32.0,32.0
HC-2019-0024030,1,23.000000,23.0,23.0
HC-2019-0028374,1,71.000000,71.0,71.0
HC-2019-0040050,1,43.000000,43.0,43.0
HC-2019-0047871,1,12.000000,12.0,12.0
HC-2019-0056051,1,35.000000,35.0,35.0
HC-2019-0064407,1,42.000000,42.0,42.0
HC-2019-0068020,1,34.000000,34.0,34.0


**Observaciones**

La variable `edad_victima_num` puede agregarse razonablemente a nivel siniestro mediante una edad promedio de las víctimas asociadas.

Dado que el 89,72% de los siniestros presenta una única víctima, la edad promedio coincide en la mayoría de los casos con la edad real registrada.

Por este motivo, se considera viable incorporar una variable agregada de edad promedio al dataset principal.

In [ ]:
# Analizamos qué combinaciones de sexo aparecen por siniestro

combinacion_sexos = (
    victimas
    .groupby("id_siniestro")["sexo_victima"]
    .apply(
        lambda x: tuple(
            sorted(
                x.astype(str)
                 .str.upper()
                 .unique()
            )
        )
    )
)

combinacion_sexos.value_counts().head(15)

sexo_victima
(M,)          32036
(SD,)         14988
(F,)          14733
(F, M)         2727
(M, SD)         684
(F, SD)         472
(F, M, SD)      178
Name: count, dtype: int64

**Observaciones**

El 93,83% de los siniestros presenta una única categoría de sexo registrada (`M`, `F` o `SD`).

Solamente el 6,17% de los siniestros contiene combinaciones de categorías que requieren un tratamiento especial.

Dado que la variable presenta una alta cobertura y una estructura mayormente simple, se considera viable su incorporación al dataset principal mediante una agregación a nivel siniestro.

### **Construcción de variables agregadas provenientes de Víctimas**

Las variables de la tabla Víctimas se agregan a nivel siniestro para poder incorporarlas al dataset principal.
Se construyen:
- edad_promedio_victimas
- sexo_victima_agregado

In [31]:
# Edad promedio por siniestro

edad_agregada = (
    victimas
    .groupby("id_siniestro")["edad_victima_num"]
    .mean()
    .reset_index()
    .rename(
        columns={
            "edad_victima_num": "edad_promedio_victimas"
        }
    )
)

# Sexo agregado por siniestro

def agregar_sexo(grupo):

    sexos = set(
        grupo.astype(str)
             .str.upper()
             .str.strip()
    )

    if sexos == {"M"}:
        return "M"

    elif sexos == {"F"}:
        return "F"

    elif sexos == {"SD"}:
        return "SD"

    elif sexos == {"M", "SD"}:
        return "M"

    elif sexos == {"F", "SD"}:
        return "F"

    else:
        return "MIXTO"


sexo_agregado = (
    victimas
    .groupby("id_siniestro")["sexo_victima"]
    .apply(agregar_sexo)
    .reset_index()
    .rename(
        columns={
            "sexo_victima": "sexo_victima_agregado"
        }
    )
)

# Revisamos los resultados

print("Edad:")
print(edad_agregada.head())

print("\nSexo:")
print(sexo_agregado.head())

Edad:
      id_siniestro  edad_promedio_victimas
0  HC-2019-0014291                    25.0
1  HC-2019-0016448                    32.0
2  HC-2019-0024030                    23.0
3  HC-2019-0028374                    71.0
4  HC-2019-0040050                    43.0

Sexo:
      id_siniestro sexo_victima_agregado
0  HC-2019-0014291                     M
1  HC-2019-0016448                     M
2  HC-2019-0024030                     M
3  HC-2019-0028374                     M
4  HC-2019-0040050                     M


### **Incorporación de variables agregadas al dataset principal**

Las variables derivadas a partir de la tabla Víctimas se incorporan a la tabla Hechos utilizando `id_siniestro` como clave de unión.

De esta forma se mantiene una única unidad de análisis a nivel siniestro.

In [35]:
# Incorporamos edad promedio

hechos = hechos.merge(
    edad_agregada,
    on="id_siniestro",
    how="left"
)

# Incorporamos sexo agregado

hechos = hechos.merge(
    sexo_agregado,
    on="id_siniestro",
    how="left"
)

# Verificamos resultado

print("Shape:")
print(hechos.shape)

print("\nÚltimas columnas:")
print(hechos.columns.tolist()[-5:])

Shape:
(65818, 29)

Últimas columnas:
['sexo_victima_agregado_x', 'edad_promedio_victimas_y', 'sexo_victima_agregado_y', 'edad_promedio_victimas', 'sexo_victima_agregado']


**Observaciones**

La incorporación de las variables agregadas desde Víctimas se realizó correctamente.

El dataset mantiene 65.818 registros, por lo que no se generaron duplicaciones ni pérdidas de siniestros durante la unión.

Las nuevas variables incorporadas son `edad_promedio_victimas` y `sexo_victima_agregado`.

### **Construcción del dataset principal de trabajo**

A partir de las transformaciones realizadas, se construye el dataframe principal `df`.

Este dataset conserva variables de soporte, variables predictoras candidatas y la variable objetivo. La selección definitiva para modelado se realizará en una etapa posterior.

In [36]:
# Construimos el dataframe principal de trabajo para las próximas etapas

columnas_df = [
    "id_siniestro",
    "siniestro_fatal",
    "fecha_siniestro",
    "anio_siniestro",
    "mes_siniestro",
    "dia_siniestro",
    "dia_semana",
    "rango_horario",
    "comuna_siniestro",
    "tipo_de_via_siniestro",
    "participantes_siniestro",
    "modo_desplazamiento_victima",
    "contraparte_siniestro",
    "latitud_siniestro",
    "longitud_siniestro",
    "edad_promedio_victimas",
    "sexo_victima_agregado"
]

df = hechos[columnas_df].copy()

# Verificamos dimensiones y columnas del dataset resultante

print("Shape:")
print(df.shape)

print("\nColumnas:")
print(df.columns.tolist())

Shape:
(65818, 17)

Columnas:
['id_siniestro', 'siniestro_fatal', 'fecha_siniestro', 'anio_siniestro', 'mes_siniestro', 'dia_siniestro', 'dia_semana', 'rango_horario', 'comuna_siniestro', 'tipo_de_via_siniestro', 'participantes_siniestro', 'modo_desplazamiento_victima', 'contraparte_siniestro', 'latitud_siniestro', 'longitud_siniestro', 'edad_promedio_victimas', 'sexo_victima_agregado']


**Observaciones**

Luego de las transformaciones realizadas, se construyó el dataframe principal `df`, compuesto por 65.818 siniestros y 17 variables.

El dataset integra información proveniente de las tablas Hechos y Víctimas, manteniendo una única fila por siniestro y conservando tanto variables descriptivas como la variable objetivo del proyecto.

### **Validación final del dataset ETL**

Se realizan verificaciones finales sobre el dataset resultante para confirmar la integridad de los registros, identificar valores faltantes y revisar los tipos de datos de las variables disponibles.

In [37]:
# Verificamos duplicados por siniestro

print("Duplicados por id_siniestro:")
print(df["id_siniestro"].duplicated().sum())

print("\n" + "="*60 + "\n")

# Verificamos valores faltantes

print("Valores faltantes:")
print(df.isna().sum().sort_values(ascending=False))

print("\n" + "="*60 + "\n")

# Revisamos tipos de datos

print("Tipos de datos:")
print(df.dtypes)

Duplicados por id_siniestro:
0


Valores faltantes:
edad_promedio_victimas         20559
tipo_de_via_siniestro          12227
longitud_siniestro              3031
latitud_siniestro               3031
comuna_siniestro                3017
rango_horario                     77
contraparte_siniestro              0
modo_desplazamiento_victima        0
participantes_siniestro            0
id_siniestro                       0
siniestro_fatal                    0
dia_semana                         0
dia_siniestro                      0
mes_siniestro                      0
anio_siniestro                     0
fecha_siniestro                    0
sexo_victima_agregado              0
dtype: int64


Tipos de datos:
id_siniestro                           object
siniestro_fatal                         int32
fecha_siniestro                datetime64[ns]
anio_siniestro                          int64
mes_siniestro                           int64
dia_siniestro                           int64
dia_semana  

**Observaciones finales de la fase ETL**

El dataset principal fue construido correctamente y no presenta registros duplicados.

Persisten valores faltantes en algunas variables, especialmente en `edad_promedio_victimas` y `tipo_de_via_siniestro`.

Dado que el tratamiento de dichos faltantes depende de los resultados del análisis exploratorio y de las estrategias de modelado seleccionadas, su evaluación e imputación se abordarán en las siguientes etapas del proyecto.

## **Conclusiones de la fase ETL**

Durante esta etapa se integraron las tablas HECHOS y VICTIMAS utilizando `id_siniestro` como clave de unión y se construyó una variable objetivo binaria (`siniestro_fatal`) a nivel siniestro.

También se realizaron transformaciones de tipos de datos, normalización de categorías, generación de variables derivadas y agregación de información proveniente de la tabla VICTIMAS.

El dataset resultante contiene 65.818 registros y 17 variables, manteniendo una única fila por siniestro y sin registros duplicados.

Se identificaron valores faltantes en algunas variables, especialmente en `edad_promedio_victimas` y `tipo_de_via_siniestro`. Debido a que el tratamiento de estos faltantes depende del análisis exploratorio y de las estrategias de modelado seleccionadas, su evaluación se pospone para las siguientes etapas del proyecto.

El dataset queda preparado para iniciar la fase de Análisis Exploratorio de Datos (EDA).

In [38]:

# Exportamos el dataset final

df.to_csv(
    "../data_processed/siniestros_viales_modelado.csv",
    index=False
)

df.to_excel(
    "../data_processed/siniestros_viales_modelado.xlsx",
    index=False
)

print("Archivos exportados correctamente.")

Archivos exportados correctamente.
